# StarCoder2 Fine-tuning (Unsloth)
Set `SETUP = "A"` or `"B"` in the config cell, then Run All. Trains one setup per run.


In [1]:
# Force single GPU (avoids cuda:0/cuda:1 device-mismatch error)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [2]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/

## Install Unsloth only. It pins compatible transformers/peft/trl/bitsandbytes.
# Do NOT separately install trl/transformers - that causes the _LazyModule conflict.
!pip install -q unsloth


In [3]:
# ===== CONFIG: choose which setup to train =====
SETUP = "A"          # "A" = direct (code->comment),  "B" = chain-of-thought (code->reasoning->comment)
NUM_EPOCHS = 2
MAX_SEQ_LEN = 1280
OUTPUT_DIR = f"outputs_setup_{SETUP.lower()}"
ADAPTER_DIR = f"setup_{SETUP.lower()}_adapters"
print(f"Training Setup {SETUP} | epochs={NUM_EPOCHS} | max_len={MAX_SEQ_LEN}")


Training Setup A | epochs=2 | max_len=1280


In [4]:
# Load StarCoder2-3B in 4-bit via Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = "bigcode/starcoder2-3b",
    max_seq_length = MAX_SEQ_LEN,
    dtype        = None,
    load_in_4bit = True,
)
print("Model loaded.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Starcoder2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Unsloth: bigcode/starcoder2-3b has no pad_token. Using pad_token = <fim_pad>.
Model loaded.


In [5]:
# Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA attached.")


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


LoRA attached.


In [6]:
# Load the data file for the chosen setup
import json
BASE = "/kaggle/input/datasets/ayushsrivastava1805/python-train-cot"

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

if SETUP == "A":
    data = load_jsonl(f"{BASE}/python_train_filtered.jsonl")
else:
    data = load_jsonl(f"{BASE}/python_train_cot.jsonl")

print(f"Setup {SETUP}: {len(data):,} samples | fields: {list(data[0].keys())}")


Setup A: 22,389 samples | fields: ['oldf', 'patch', 'msg', 'id', 'y', 'lang_pred']


In [7]:
# Format helpers (remove leading license header so model sees real code)
def strip_license_header(code_str):
    lines = code_str.split("\n")
    start = 0
    for i, line in enumerate(lines):
        s = line.strip()
        if s.startswith("#") or s == "" or s.startswith('\"\"\"') or s.startswith("'''"):
            start = i + 1
        else:
            break
    if start >= len(lines) - 2:
        return code_str
    return "\n".join(lines[start:])

def format_sample(s):
    oldf  = strip_license_header(s.get("oldf", ""))[:1500]
    patch = s.get("patch", "")[:800]
    msg   = s.get("msg", "")
    if SETUP == "A":
        text = f"### Code:\n{oldf}\n\n### Change:\n{patch}\n\n### Review Comment:\n{msg}{tokenizer.eos_token}"
    else:
        reasoning = s.get("reasoning", "")
        text = f"### Code:\n{oldf}\n\n### Change:\n{patch}\n\n### Reasoning:\n{reasoning}\n\n### Review Comment:\n{msg}{tokenizer.eos_token}"
    return {"text": text}

print("--- One formatted example ---")
print(format_sample(data[0])["text"][:700])


--- One formatted example ---
### Code:
For internal use only; no backwards-compatibility guarantees.
"""

import sys
import traceback

import six

from apache_beam.internal import util
from apache_beam.pvalue import TaggedOutput
from apache_beam.transforms import DoFn
from apache_beam.transforms import core
from apache_beam.transforms.core import RestrictionProvider
from apache_beam.transforms.window import GlobalWindow
from apache_beam.transforms.window import TimestampedValue
from apache_beam.transforms.window import WindowFn
from apache_beam.utils.windowed_value import WindowedValue


class NameContext(object):
  """Holds the name information for a step."""

  def __init__(self, step_name):
    """Creates a new step 


In [8]:
# Build the full dataset
from datasets import Dataset
formatted = [format_sample(s) for s in data]
dataset = Dataset.from_list(formatted)
print(f"Dataset ready: {len(dataset):,} samples")


Dataset ready: 22,389 samples


In [9]:
# Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 25,
        save_steps = 500,
        save_total_limit = 6,
        output_dir = OUTPUT_DIR,
        optim = "adamw_8bit",
        seed = 42,
        lr_scheduler_type = "cosine",
        dataset_text_field = "text",
        max_steps = 5500
    ),
)

trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/22389 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 22,389 | Num Epochs = 2 | Total steps = 5,500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 9,093,120 of 3,039,464,448 (0.30% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
25,3.260808
50,2.426780
75,2.015950
100,1.832554
125,1.701586
150,1.693583
175,1.675117
200,1.630524
225,1.624484
250,1.583192


Unsloth: Restored added_tokens_decoder metadata in outputs_setup_a/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_setup_a/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_setup_a/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_setup_a/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_setup_a/checkpoint-2500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_setup_a/checkpoint-3000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_setup_a/checkpoint-3500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_setup_a/checkpoint-4000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_setup_a/checkpoint-4500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata

TrainOutput(global_step=5500, training_loss=1.344950558055531, metrics={'train_runtime': 39182.4893, 'train_samples_per_second': 1.123, 'train_steps_per_second': 0.14, 'total_flos': 4.5983882806004736e+17, 'train_loss': 1.344950558055531, 'epoch': 1.96507369361322})

In [10]:
# Save the trained LoRA adapters
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Done. Adapters saved to '{ADAPTER_DIR}'")


Unsloth: Restored added_tokens_decoder metadata in setup_a_adapters/tokenizer_config.json.


Done. Adapters saved to 'setup_a_adapters'
